# Running the CESM2 diffusion emulator

Load a checkpoint, generate a climate scenario, save it, plot it.

This notebook is **self-contained and portable**: every path is a variable in the
first cell, and it runs anywhere the repository's dependencies are installed —
laptop, workstation, or an HPC compute node. There is a CPU fallback, so you can
smoke-test the whole pipeline without a GPU.

---

## What the emulator is

A **conditional video diffusion model**. It denoises a `(batch, channels, 1, 192, 288)`
field conditioned on gridded forcing maps.

- **Conditioning (input)**: up to 3 channels — cumulative CO₂, sulphate (SUL),
  black carbon (BC) — one `192×288` map per year
- **Target (output)**: 1 or 2 channels — `TREFHT` (surface air temperature) and,
  in two-channel checkpoints, `PRECT` (precipitation)
- **One sample = one year.** A scenario is generated year by year; an ensemble
  member is one random seed.

So running it means: **load → build conditioning → sample → denormalise**.

## What you need

| | | |
|---|---|---|
| checkpoint | `*.pt` | holds `EMA` weights, the `PCA` basis and `COND_NORM` ranges |
| model config | `configs/config_aero.yaml` | defines the UNet and noise scheduler |
| data config | `configs/config_data_*.yaml` | names the cond/target variables and PCA settings |
| conditioning file | `emissions_*.nc` | CO2/SUL/BC maps per year |

A CESM2 reference is needed only if you want to *score* the output; generating
does not require it.

---
# Step 1 — Configure

**Edit this cell.** Everything downstream reads these variables, and nothing
else in the notebook contains a machine-specific path. Each can also be supplied
through an environment variable of the same name.

In [ ]:
# ── EDIT ME ───────────────────────────────────────────────────────────────────
import os
from pathlib import Path

# Repository root, found by walking up from the working directory until the
# repo's marker files appear. Works whether the kernel starts in <repo>/ or in
# <repo>/notebooks/, which differs between Jupyter, VS Code and papermill.
def _find_repo(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "main_aero.py").exists() and (p / "configs").is_dir():
            return p
    raise FileNotFoundError(
        f"repository root not found above {start} — set EMULATOR_REPO")

REPO_DIR = Path(os.environ["EMULATOR_REPO"]).resolve() if os.environ.get("EMULATOR_REPO") \
           else _find_repo(Path.cwd().resolve())

# The trained model.
CKPT       = Path(os.environ.get("EMULATOR_CKPT", REPO_DIR / "runs/run_mseyb_BCprect_860.pt"))

# Configs (relative to REPO_DIR unless absolute).
MODEL_CFG  = Path(os.environ.get("EMULATOR_MODEL_CFG", REPO_DIR / "configs/config_aero.yaml"))
DATA_CFG   = Path(os.environ.get("EMULATOR_DATA_CFG",
                                 REPO_DIR / "configs/config_data_ybias_BCprect.yaml"))

# Conditioning file for the scenario you want to generate.
COND_FILE  = Path(os.environ.get("EMULATOR_COND",
                                 REPO_DIR / "data/emissions_ssp370_only_timefixed_bc_co2fix.nc"))

# Where to write output.
OUT_DIR    = Path(os.environ.get("EMULATOR_OUT", REPO_DIR / "notebook_output"))

# ── Run size. Lower these to smoke-test on a CPU or a small GPU. ──────────────
N_YEARS      = int(os.environ.get("EMULATOR_N_YEARS", 0))   # 0 = every year in COND_FILE
SAMPLE_STEPS = int(os.environ.get("EMULATOR_STEPS", 50))    # 50 = production; 10 = quick look
BATCH_SIZE   = int(os.environ.get("EMULATOR_BATCH", 16))    # lower if you run out of memory
SEED         = 1234

OUT_DIR.mkdir(parents=True, exist_ok=True)

# The repo must be BOTH the working directory (Hydra resolves config paths
# relative to it) and on sys.path (so `import eval_aero` works). chdir alone is
# not enough: a kernel started in notebooks/ has notebooks/ on sys.path, not the
# repo, and the import fails.
import sys
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

for label, p in [("repo", REPO_DIR), ("checkpoint", CKPT), ("model config", MODEL_CFG),
                 ("data config", DATA_CFG), ("conditioning", COND_FILE)]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {label:14s} {p}")
print(f"\noutput -> {OUT_DIR}")

---
# Step 2 — Preflight

Checks the dependencies and picks a device. Nothing is loaded yet, so a failure
here is cheap.

If packages are missing:

```bash
pip install torch diffusers accelerate hydra-core omegaconf \
            xarray netcdf4 numpy scikit-learn ema-pytorch tqdm matplotlib cartopy
```

`cartopy` is needed only because `eval_aero` imports it at module level for its
own map plotting; this notebook does not otherwise use it.

In [ ]:
import importlib.util, sys

REQUIRED = ["torch", "xarray", "numpy", "omegaconf", "hydra", "diffusers",
            "sklearn", "ema_pytorch", "tqdm", "matplotlib", "cartopy", "scipy"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
print("missing:", missing if missing else "none")

import torch
# WHICH torch got imported. A half-written or wrong-architecture install in
# ~/.local shadows everything else, because a plain interpreter puts the user
# site-packages first; the failure then surfaces much later as a missing .so.
# Set PYTHONNOUSERSITE=1 before starting the kernel to ignore ~/.local.
print(f"torch   {torch.__version__}")
print(f"        {torch.__file__}")

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE  = torch.bfloat16          # ~2x faster; matches how the evaluations were run
    print(f"device: cuda -> {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    DTYPE  = torch.float32           # bf16 on CPU is slow and poorly supported
    print("device: CPU — this works but is SLOW.")
    print("        Set N_YEARS=2 and SAMPLE_STEPS=10 above for a smoke test.")
print(f"dtype:  {DTYPE}")

---
# Step 3 — Load the checkpoint

`load_model()` returns the **EMA** weights (not the raw ones) plus the PCA state,
and injects the checkpoint's `COND_NORM` clip ranges into the dataset module so
that inference normalises conditioning exactly as training did.

> ⚠️ Watch the printed output. `[COND-NORM] using checkpoint-persisted clip ranges`
> is what you want. `WARNING: checkpoint has no COND_NORM` means the run is
> **miscalibrated**, not merely approximate — stop and use a checkpoint that has it.

In [ ]:
from omegaconf import OmegaConf
from hydra.utils import instantiate
import lumi_paths as L                      # pure path helper; safe to import anywhere
from eval_aero import load_model, build_cond_tensor, generate_timeseries

model, pca_state = load_model(str(CKPT), str(MODEL_CFG), DEVICE)

cfg       = L.resolve_cfg(OmegaConf.load(str(MODEL_CFG)))
scheduler = instantiate(cfg.scheduler)               # ContinuousDDPM
OUT_CH    = int(cfg.model.get("out_channels", 1))

print(f"\nPCA state : {None if pca_state is None else list(pca_state.keys())}")
print(f"out_channels: {OUT_CH}  (1 = TREFHT only, 2 = TREFHT + PRECT)")
print(f"parameters : {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

---
# Step 4 — Build the conditioning tensor

**The step that matters most.** Smoothing and PCA projection must mirror the
training pipeline exactly. Feed the model raw inventory fields it never saw and
it imprints grid-scale texture — shipping lanes, flight paths — onto the output.

`pca_objects` decides the basis:

| value | meaning | use when |
|---|---|---|
| `pca_state["cond"]` | apply the **persisted** basis | the scenario was in training |
| `"fit"` | fit a **fresh** basis on this cond | an unseen scenario |
| `None` | skip PCA entirely | almost never |

In [ ]:
data_cfg  = L.resolve_cfg(OmegaConf.load(str(DATA_CFG)))
cond_vars = list(data_cfg.cond_vars)

cond_tensor, years, lat, lon = build_cond_tensor(
    cond_file          = str(COND_FILE),
    cond_vars          = cond_vars,
    time_dim           = "time",
    pca_objects        = pca_state.get("cond") if pca_state else None,
    n_components_cond  = list(data_cfg.n_components_cond),
    cond_smooth_sigma  = list(data_cfg.cond_smooth_sigma),
    cond_smooth_method = data_cfg.get("cond_smooth_method", "gaussian"),
)

if N_YEARS:                                   # optional truncation for a quick run
    cond_tensor, years = cond_tensor[:, :N_YEARS], years[:N_YEARS]

print(f"cond vars : {cond_vars}")
print(f"cond shape: {tuple(cond_tensor.shape)}  = (n_vars, T, H, W)")
print(f"years     : {years.min()}-{years.max()}  ({len(years)} of them)")

---
# Step 5 — Sample

One diffusion run over every year in the conditioning tensor.

- `seed` selects the **ensemble member** — same conditioning, different seed
- `target_channel=None` returns all channels, so one pass gives both variables
- `guidance_* = 1.0` everywhere means direct conditioning and **one** forward
  pass. Any other value switches on per-channel classifier-free guidance and
  costs four passes per step.

Rough cost: about **1 GPU-hour per 200 scenario-years per member** at 50 steps.

In [ ]:
import time
t0 = time.time()

gen_norm = generate_timeseries(
    model          = model,
    scheduler      = scheduler,
    cond_tensor    = cond_tensor,
    device         = DEVICE,
    dtype          = DTYPE,
    sample_steps   = SAMPLE_STEPS,
    batch_size     = BATCH_SIZE,
    seed           = SEED,
    guidance_co2   = 1.0,
    guidance_sul   = 1.0,
    guidance_bc    = 1.0,
    out_channels   = OUT_CH,
    target_channel = None,          # all channels
)

print(f"\n{gen_norm.shape} = (T, C, H, W), in NORMALISED model space")
print(f"{time.time()-t0:.1f} s for {len(years)} years at {SAMPLE_STEPS} steps")

---
# Step 6 — Denormalise

The sampler returns normalised space. Each target channel has its own inverse
transform: `TREFHT` → °C, `PRECT` → mm/day.

In [ ]:
import numpy as np
from data.climate_dataset import DENORM_FN

target_vars = list(data_cfg.target_vars)[:OUT_CH]
fields = {v: DENORM_FN[v](gen_norm[:, i]) for i, v in enumerate(target_vars)}

# cos(lat) weighting: an unweighted global mean over this grid is badly wrong,
# because the poles are massively over-sampled on a regular lat/lon grid.
w = np.cos(np.deg2rad(lat))[None, :, None]
gmeans = {v: (a * w).sum(axis=(1, 2)) / (w.sum() * a.shape[2]) for v, a in fields.items()}

for v, g in gmeans.items():
    print(f"{v:7s} {fields[v].shape}  global mean {g[0]:.3f} ({years[0]}) "
          f"-> {g[-1]:.3f} ({years[-1]})")

---
# Step 7 — Save

A plain NetCDF with real coordinates, readable by anything.

In [ ]:
import xarray as xr

UNITS = {"TREFHT": "degC", "PRECT": "mm/day"}
ds_out = xr.Dataset(
    {v: (("year", "lat", "lon"), a.astype("float32"),
         {"units": UNITS.get(v, ""), "long_name": f"emulated {v}"})
     for v, a in fields.items()},
    coords={"year": years, "lat": lat, "lon": lon},
    attrs={"checkpoint": CKPT.name, "cond_file": COND_FILE.name,
           "sample_steps": SAMPLE_STEPS, "seed": SEED},
)
path = OUT_DIR / f"emulated_{COND_FILE.stem}_seed{SEED}.nc"
ds_out.to_netcdf(path)
print(f"wrote {path}  ({path.stat().st_size/1e6:.1f} MB)")
ds_out

---
# Step 8 — Look at it

Deliberately plain matplotlib — no cartopy — so this renders anywhere.

In [ ]:
import matplotlib.pyplot as plt

n = len(fields)
fig, axes = plt.subplots(n, 2, figsize=(12, 3.6 * n), squeeze=False,
                         gridspec_kw={"width_ratios": [1, 1.35]})
for i, (v, a) in enumerate(fields.items()):
    axes[i][0].plot(years, gmeans[v], lw=2, color="#D55E00")
    axes[i][0].set(xlabel="year", ylabel=f"{v} [{UNITS.get(v,'')}]",
                   title=f"{v}: global mean")
    axes[i][0].grid(alpha=0.3)

    im = axes[i][1].pcolormesh(lon, lat, a[-1], shading="auto",
                               cmap="RdBu_r" if v == "TREFHT" else "BrBG")
    axes[i][1].set(xlabel="lon", ylabel="lat", title=f"{v}: year {years[-1]}")
    plt.colorbar(im, ax=axes[i][1], label=UNITS.get(v, ""))
fig.tight_layout()
plt.show()

---
# Step 9 — An ensemble is a loop over seeds

The spread across members is the model's internal variability. Each member costs
a full sampling pass, which is why large ensembles are batch jobs rather than
notebook work.

In [ ]:
members = {}
for seed in (SEED, SEED + 1, SEED + 2):
    g = generate_timeseries(model, scheduler, cond_tensor, DEVICE, DTYPE,
                            sample_steps=SAMPLE_STEPS, batch_size=BATCH_SIZE,
                            seed=seed, out_channels=OUT_CH, target_channel=0)
    members[seed] = DENORM_FN[target_vars[0]](g)

M = np.stack(list(members.values()))                       # (member, T, H, W)
G = (M * w[None]).sum(axis=(2, 3)) / (w.sum() * M.shape[3])

plt.figure(figsize=(7, 4))
for s, g in zip(members, G):
    plt.plot(years, g, lw=1, alpha=0.8, label=f"seed {s}")
plt.plot(years, G.mean(0), lw=2.5, color="k", label="ensemble mean")
plt.xlabel("year"); plt.ylabel(f"{target_vars[0]} [{UNITS.get(target_vars[0],'')}]")
plt.legend(frameon=False); plt.grid(alpha=0.3); plt.title("ensemble spread")
plt.show()

print(f"inter-member sd of the final year: {M[:, -1].std(axis=0).mean():.4f}")

---
# Going further

## Full evaluation against CESM2

`eval_aero.py` does what this notebook does, for every scenario and many
members, and additionally scores the result against a CESM2 reference and writes
NetCDF plus summary plots:

```bash
python eval_aero.py \
    --checkpoint runs/<checkpoint>.pt \
    --output-dir <output directory> \
    --data-config configs/config_data_ybias_BCprect.yaml \
    --experiments ssp370 \
    --members 5 \
    --sample-steps 50
```

Useful flags: `--members`, `--experiments`, `--sample-steps`, `--target-var`,
`--fp32`, and `--shard-rank` / `--n-shards` to split experiments across
processes. It needs the CESM2 reference data the experiment table points at.

## Training

`main_aero.py` is a [Hydra](https://hydra.cc) app — the config *is* the
interface, and any key can be overridden on the command line:

```bash
python main_aero.py --config-name=config_aero.yaml \
    trainer.hyperparameters.save_dir=runs/my_run \
    trainer.hyperparameters.save_every=140
```

Multi-GPU goes through `accelerate`:

```bash
accelerate launch --num_processes <N> main_aero.py --config-name=config_aero.yaml
```

The three settings that decide whether a run is usable:

| key | meaning | trap |
|---|---|---|
| `save_every` | checkpoint interval in **optimizer steps** | too large and a long job saves nothing |
| `load_path` | `0` fresh, `"newest"` resume, or an explicit `.pt` | resuming mid-epoch has deadlocked before |
| `reset_optimizer` | keep Adam momentum across chained jobs | must stay `false` for a continuation |

To verify a resume is healthy: the first epoch is a fast-forward through the
dataloader and finishes suspiciously quickly; the **second** must run at a normal
step time, and a checkpoint must appear within `save_every` optimizer steps.
Silence with the GPU pinned at 100% is a deadlock, not progress.

---
# Gotchas

**Conditioning — where the subtle failures live**
1. `COND_NORM` missing from a checkpoint means the run is **miscalibrated**, not approximate. The loader warns; do not ignore it.
2. Apply the persisted PCA basis for trained scenarios and `"fit"` for unseen ones. Skipping PCA feeds full-rank conditioning the model never saw.
3. `guidance_* = 1.0` is one forward pass; anything else costs four. Guidance tuning at high forcing does **not** work — the response is sub-additive.

**Reading the output**
4. Always cos(lat)-weight global means. Unweighted means over this grid are wrong by degrees, not decimals.
5. Precipitation is roughly 10× noisier relative to its forced signal than temperature. A comparison that shows "no significant difference" is usually a **detection limit**, not a pass.
6. The one-step training loss is a poor proxy for sampled-field quality — evaluation metrics plateau long before it does. Judge a checkpoint on sampled output.

**Scale**
7. One member of a 200-year scenario is roughly one GPU-hour at 50 sampling steps. Ensembles are batch jobs.
8. Reduce `BATCH_SIZE` first if you run out of memory; it does not change the result, only the memory footprint.

---
# Where to look next

| you want | file |
|---|---|
| the model | `models/video_net.py` (`UNetModel3D`) |
| the training loop | `trainer/unetTrainer.py` |
| normalisation, PCA, the dataset | `data/climate_dataset.py` |
| evaluation end to end | `eval_aero.py` |
| figures | `scripts/paper_fig_*.py` |